# Training data collection, preprocessing, training, and validation

This notebook only sets variables, calls `.py` functions, and displays tables/plots.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'acquisition.py').exists():
    ROOT = Path(r'D:/BME/BCI/online_bci/online_eeg')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from acquisition import AcquisitionConfig, collect_training_segments
from preprocessing import AudioLabelConfig, PreprocessConfig, preprocess_many_recordings
from training import TrainingConfig, WindowConfig, train_validate_pipeline
from plots import plot_labeled_recording

print('Pipeline root:', ROOT)


## Settings

In [ ]:
RAW_DIR = ROOT / 'data' / 'raw_training'
LABELED_DIR = ROOT / 'data' / 'labeled_training'
RUN_DIR = ROOT / 'runs' / 'run_001'

COLLECT_TRAINING = False
TRAIN_SEGMENT_SEC = 300.0
TRAIN_SEGMENT_NAMES = ('train_01', 'train_02')
TRAIN_RAW_NPZ = []

ACQ = AcquisitionConfig(
    samplerate=200,
    channels=(1, 2, 3, 4, 5),
    chunk_sec=TRAIN_SEGMENT_SEC,
)

PRE = PreprocessConfig(
    eeg_channels=(1, 2, 3, 4),
    audio_channel=5,
    bandpass_low_hz=1.0,
    bandpass_high_hz=40.0,
    notch_hz=(60.0,),
    notch_quality_factor=30.0,
    filter_order=4,
    demean_channels=True,
)

LABELS = AudioLabelConfig(
    class_names=('Eyes Closed', 'Eyes Open'),
    baseline_label=0,
    active_label=1,
    cue_label_sequence=None,
    alternate_binary_labels=True,
    label_duration_sec=None,  # transition mode: each cue switches state until the next cue.
    label_start_offset_sec=0.0,  # label switch starts exactly at cue onset.
    envelope_window_sec=0.025,
    onset_threshold=None,
    onset_min_interval_sec=0.50,
)

WIN = WindowConfig(
    feature_mode='fft_bandpower',
    window_sec=1.0,
    stride_sec=0.10,
    label_mode='endpoint',
    bandpower_hz=(18.0, 22.0),
)

TRAIN = TrainingConfig(
    train_fraction=0.80,
    hidden_size=32,
    num_layers=1,
    dropout=0.0,
    batch_size=64,
    epochs=20,
    lr=1e-3,
    seed=888,
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


## Collect 5-minute training segments

In [ ]:
if COLLECT_TRAINING:
    train_raw_paths = collect_training_segments(
        output_dir=RAW_DIR,
        segment_names=TRAIN_SEGMENT_NAMES,
        config=ACQ,
        duration_sec=TRAIN_SEGMENT_SEC,
    )
else:
    train_raw_paths = [Path(p) for p in TRAIN_RAW_NPZ]

if not train_raw_paths:
    raise ValueError('Set COLLECT_TRAINING=True or populate TRAIN_RAW_NPZ with existing raw .npz files.')

train_raw_paths


## Preprocess, FFT-capable windowing inputs, and audio-channel labels

In [ ]:
labeled_paths = preprocess_many_recordings(
    raw_npz_paths=train_raw_paths,
    output_dir=LABELED_DIR,
    preprocess_config=PRE,
    label_config=LABELS,
)

labeled_paths


In [ ]:
fig, ax = plot_labeled_recording(labeled_paths[0], max_duration_sec=None)


## Train and validate LSTM

In [ ]:
train_result = train_validate_pipeline(
    labeled_npz_paths=labeled_paths,
    output_dir=RUN_DIR,
    window_config=WIN,
    training_config=TRAIN,
)

checkpoint_path = train_result['checkpoint_path']
print('Saved model checkpoint:', checkpoint_path)
checkpoint_path


In [ ]:
history = train_result['history']
display(history.tail())
display(train_result['validation_summary'])
display(train_result['validation_per_class'])

ax = history.plot(x='epoch', y=['train_acc', 'val_acc'], marker='o', figsize=(8, 4))
ax.set_ylabel('Accuracy')
ax.set_title('Training and validation accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()
